# Test fat tree with fixed K

This section executes a single, focused run of the ECLYPSE+NET framework on a specific Fat-Tree topology (in this case, $k=6$). 

In [1]:
import time
import pandas as pd
import random
import numpy as np

GLOBAL_SEED = 42

random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)

# Import of the base Eclypse modules
from eclypse.graph import Infrastructure
from eclypse.graph import Application
from eclypse.simulation import Simulation, SimulationConfig
from eclypse.placement.strategies import StaticStrategy

# Import of the Eclypse+NET modules
from eclypse.network import Network, NetworkApplication, PacketGenerationEvent, RoutingEvent, RoutingMetric

# Import the topology generator
from eclypse.builders.infrastructure.generators import get_fat_tree


def run_eclypse_net_shared(base_infra: Infrastructure, steps: int = 10) -> float:
    """Execute the Eclypse+NET simulation by parsing a shared topology."""
    num_nodes = len(base_infra.nodes)
    infra = Network(f"Net_Infra_{num_nodes}")
    app = NetworkApplication(f"Net_App_{num_nodes}")
    mapping = {}

    # Parsing the nodes of the base infrastructure
    for node in base_infra.nodes():
        if "host" in node:
            infra.add_host(node, processing_time=0.0001, cpu=2, ram=4)
            app_name = f"App_{node}"
            app.add_node(app_name, cpu=1, ram=1)
            mapping[app_name] = node
        else:
            # Core, aggregation, and edge switches act as routers
            infra.add_router(node, processing_time=0.0001)

    # Parsing of the edges of the base infrastructure
    for u, v in base_infra.edges():
        infra.add_edge(u, v, bandwidth_mbps=1000, length_km=1)

    # Generation of the traffic using the 'hosts' property of the infrastructure.
    # Convert the set of hosts to a list to access easily the hosts [0] e [1:]
    available_hosts = list(infra.hosts)

    if len(available_hosts) > 1:
        source_app = f"App_{available_hosts[0]}"
        for target_node in available_hosts[1:]:
            target_app = f"App_{target_node}"
            app.add_edge(source_app, target_app, packet_size_bytes=1000, avg_packets_per_step=0.1)

    packet_evt = PacketGenerationEvent()
    routing_evt = RoutingEvent(step_duration_s=0.001)
    metric = RoutingMetric()

    config = SimulationConfig(
        seed=GLOBAL_SEED,
        max_steps=steps,
        events=[packet_evt, routing_evt, metric],
        path="./results",
        step_every_ms=500,
        include_default_metrics=False,
        report_format="json",
        report_backend="pandas",
        remote=False
    )
    sim = Simulation(infra, simulation_config=config)
    sim.register(app, placement_strategy=StaticStrategy(mapping))

    start_time = time.perf_counter()
    sim.start()
    sim.wait()
    return time.perf_counter() - start_time


# K parameter for the Fat-Tree topology (it must be an even number)
k_single = 6

print(f"Generation of the Fat-Tree topology with k={k_single}...")
shared_topology = get_fat_tree(k=k_single, seed=GLOBAL_SEED)
total_nodes = len(shared_topology.nodes)

print(f"Total nodes in the topology: {total_nodes}")
print("Starting the simulation...")

# Execute the simulation once
execution_time = run_eclypse_net_shared(shared_topology, steps=10)

print("-" * 40)
print(f"Simulation completed successfully!")
print(f"Execution time: {execution_time:.4f} seconds")

Generation of the Fat-Tree topology with k=6...
Total nodes in the topology: 99
Starting the simulation...
15:40:56.104 | INFO | Net_App_99 - Added flow App_host_0_0_0->App_host_0_0_1, 0.1 pkt/step (avg), size 1000B
15:40:56.104 | INFO | Net_App_99 - Added flow App_host_0_0_0->App_host_0_0_2, 0.1 pkt/step (avg), size 1000B
15:40:56.104 | INFO | Net_App_99 - Added flow App_host_0_0_0->App_host_0_1_0, 0.1 pkt/step (avg), size 1000B
15:40:56.104 | INFO | Net_App_99 - Added flow App_host_0_0_0->App_host_0_1_1, 0.1 pkt/step (avg), size 1000B
15:40:56.104 | INFO | Net_App_99 - Added flow App_host_0_0_0->App_host_0_1_2, 0.1 pkt/step (avg), size 1000B
15:40:56.104 | INFO | Net_App_99 - Added flow App_host_0_0_0->App_host_0_2_0, 0.1 pkt/step (avg), size 1000B
15:40:56.104 | INFO | Net_App_99 - Added flow App_host_0_0_0->App_host_0_2_1, 0.1 pkt/step (avg), size 1000B
15:40:56.104 | INFO | Net_App_99 - Added flow App_host_0_0_0->App_host_0_2_2, 0.1 pkt/step (avg), size 1000B
15:40:56.105 | INFO |